In [0]:
# Load the 'ecommerce_transactions' table from the default schema into a Spark DataFrame
# This DataFrame will be used for further processing
# 'events' will contain all transaction records

events = spark.table("default.ecommerce_transactions")

### Create Managed Delta Table

In [0]:
# Save the DataFrame as a managed Delta table named 'events_table'
# 'delta' format enables ACID transactions and time travel
# 'overwrite' mode replaces the table if it already exists

events.write.format("delta").mode("overwrite").saveAsTable("events_table")

In [0]:
# Display the first 7 rows from the 'events_table' Delta table
# Helps verify that the table was created and data loaded correctly

display(spark.table("events_table").limit(7))

Transaction_ID,User_Name,Age,Country,Product_Category,Purchase_Amount,Payment_Method,Transaction_Date
1,Ava Hall,63,Mexico,Clothing,780.69,Debit Card,2023-04-14
2,Sophia Hall,59,India,Beauty,738.56,PayPal,2023-07-30
3,Elijah Thompson,26,France,Books,178.34,Credit Card,2023-09-17
4,Elijah White,43,Mexico,Sports,401.09,UPI,2023-06-21
5,Ava Harris,48,Germany,Beauty,594.83,Net Banking,2024-10-29
6,Elijah Harris,51,India,Toys,966.5,Cash on Delivery,2025-01-18
7,Oliver Clark,27,Germany,Home & Kitchen,341.73,Credit Card,2024-03-13


In [0]:
# Reload the Delta table into a DataFrame named 'events'
# Print the schema to see column names and types

events = spark.table("events_table")
events.printSchema()

root
 |-- Transaction_ID: long (nullable = true)
 |-- User_Name: string (nullable = true)
 |-- Age: long (nullable = true)
 |-- Country: string (nullable = true)
 |-- Product_Category: string (nullable = true)
 |-- Purchase_Amount: double (nullable = true)
 |-- Payment_Method: string (nullable = true)
 |-- Transaction_Date: date (nullable = true)



## Create incremental updates DataFrame
Simulated an incremental batch containing:

Existing Transaction_IDs → UPDATE

New Transaction_IDs → INSERT

In [0]:
# Import Row and functions for DataFrame creation and manipulation
from pyspark.sql import Row
from pyspark.sql import functions as F

# Define a list of Row objects representing updates and new inserts
# Existing Transaction_IDs (1001, 1002) will be updated
# New Transaction_IDs (99900007, 9995554) will be inserted
updates_data = [
    Row(1001, "Rahul_UPDATED", 35, "India", "Electronics", 1000.00, "Credit Card", "2022-01-01"),
    Row(1002, "Priya_UPDATED", 28, "USA", "Clothing", 500.00, "Debit Card", "2022-01-02"),
    Row(99900007, "New_User_1", 32, "Canada", "Home Appliances", 800.00, "PayPal", "2022-01-03"),
    Row(9995554, "New_User_2", 45, "UK", "Electronics", 1200.00, "Credit Card", "2022-01-04")
]

# Define column names for the DataFrame
columns = [
    "Transaction_Id",
    "User_Name",
    "Age",
    "Country",
    "Product_Category",
    "Purchase_Amount",
    "Payment_Method",
    "Transaction_Date"
]

# Create the DataFrame from the list of Rows and columns
updates_df = spark.createDataFrame(updates_data, columns)

# Convert 'Transaction_Date' column to date type for consistency
updates_df = updates_df.withColumn("Transaction_Date", F.to_date(F.col("Transaction_Date")))

# Display the updates DataFrame to verify contents
display(updates_df)

# Print the schema to check column types
updates_df.printSchema()

Transaction_Id,User_Name,Age,Country,Product_Category,Purchase_Amount,Payment_Method,Transaction_Date
1001,Rahul_UPDATED,35,India,Electronics,1000.0,Credit Card,2022-01-01
1002,Priya_UPDATED,28,USA,Clothing,500.0,Debit Card,2022-01-02
99900007,New_User_1,32,Canada,Home Appliances,800.0,PayPal,2022-01-03
9995554,New_User_2,45,UK,Electronics,1200.0,Credit Card,2022-01-04


root
 |-- Transaction_Id: long (nullable = true)
 |-- User_Name: string (nullable = true)
 |-- Age: long (nullable = true)
 |-- Country: string (nullable = true)
 |-- Product_Category: string (nullable = true)
 |-- Purchase_Amount: double (nullable = true)
 |-- Payment_Method: string (nullable = true)
 |-- Transaction_Date: date (nullable = true)



## Incremental Merge(upserts)
Used Transaction_ID as a unique business key.

In [0]:
# Import DeltaTable for performing merge operations
from delta.tables import DeltaTable

# Create a DeltaTable object for 'events_table'
deltaTable = DeltaTable.forName(spark, "events_table")

# Perform a merge (upsert) operation:
# - If Transaction_ID matches, update all columns
# - If Transaction_ID does not match, insert the new row
deltaTable.alias("t").merge(
    updates_df.alias("s"),
    "t.Transaction_ID = s.Transaction_Id"
).whenMatchedUpdateAll() \
  .whenNotMatchedInsertAll() \
  .execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

Ensuring idempotent updates and prevents duplicates

`idempotent` : denoting an element of a set which is unchanged in value when multiplied or otherwise operated on by itself.

In [0]:
# Display the row with Transaction_ID = 99900007 from 'events_table'
# This verifies that the new user was inserted during the merge

display(
    spark.table("events_table")
    .filter("Transaction_ID = 99900007")
)

Transaction_ID,User_Name,Age,Country,Product_Category,Purchase_Amount,Payment_Method,Transaction_Date
99900007,New_User_1,32,Canada,Home Appliances,800.0,PayPal,2022-01-03


In [0]:
# Display rows with Transaction_IDs 1001 and 1002 from 'events_table'
# This verifies that the existing users were updated during the merge

display(
    spark.table("events_table")
    .filter("Transaction_ID IN (1001, 1002)")
)

Transaction_ID,User_Name,Age,Country,Product_Category,Purchase_Amount,Payment_Method,Transaction_Date
1001,Rahul_UPDATED,35,India,Electronics,1000.0,Credit Card,2022-01-01
1002,Priya_UPDATED,28,USA,Clothing,500.0,Debit Card,2022-01-02


## Time Travel (Version History)

Checking Delta transaction log to identify when updates occured

In [0]:
%sql
-- Show the version history of the Delta table
-- This helps track changes and time travel
DESCRIBE HISTORY events_table;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
3,2026-02-15T19:18:18.000Z,72977528250978,samyakjain9759545711@gmail.com,MERGE,"Map(predicate -> [""(Transaction_ID#13608L = Transaction_Id#13585L)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(2384229307920853),14ad4f7b-d803-4395-84b6-7947a6d92215,0215-190144-5stx71h2-v2n,2,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 4, numTargetBytesAdded -> 9120, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 2, executionTimeMs -> 5652, materializeSourceTimeMs -> 341, numTargetRowsInserted -> 2, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 2391, numTargetRowsUpdated -> 2, numOutputRows -> 4, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 4, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 2688)",null,Databricks-Runtime/18.0.x-aarch64-photon-scala2.13
2,2026-02-15T19:03:39.000Z,72977528250978,samyakjain9759545711@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(2384229307920853),1b70e01c-4409-4e71-9c1f-bafb0e5c3add,0215-190144-5stx71h2-v2n,1,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 453352, numDeletionVectorsRemoved -> 0, numOutputRows -> 50000, numOutputBytes -> 460193)",null,Databricks-Runtime/18.0.x-aarch64-photon-scala2.13
1,2026-02-15T14:15:24.000Z,72977528250978,samyakjain9759545711@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(4192937009573805),fc73e082-af42-4a61-865b-0a087dc4e902,0215-134723-t52wdx5u-v2n,0,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 460193, numDeletionVectorsRemoved -> 0, numOutputRows -> 50000, numOutputBytes -> 453352)",null,Databricks-Runtime/18.0.x-aarch64-photon-scala2.13
0,2026-02-15T13:48:45.000Z,72977528250978,samyakjain9759545711@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(4192937009573805),53298d2f-bd23-467f-b27a-e0ee34c46d18,0215-134723-t52wdx5u-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 50000, numOutputBytes -> 460193)",null,Databricks-Runtime/18.0.x-aarch64-photon-scala2.13


Time Travel: Viewing record before MERGE (Version 10)

In [0]:
%sql
-- View the record for Transaction_ID = 1001 from an earlier version of the table
-- This demonstrates Delta Lake time travel
Select * from events_table version as of 2
WHERE Transaction_ID = 1001;

Transaction_ID,User_Name,Age,Country,Product_Category,Purchase_Amount,Payment_Method,Transaction_Date
1001,Olivia Thompson,67,Australia,Clothing,961.19,Debit Card,2023-12-26


## OPTIMIZE & ZORDER (Perfromance)

Multiple small files -> 1 optimized file

In [0]:
%sql
-- Optimize the Delta table to compact small files into larger ones
-- ZORDER by Transaction_ID improves query performance for this column
OPTIMIZE events_table
ZORDER BY (Transaction_ID);

path,metrics
,"List(1, 5, List(416012, 416012, 416012.0, 1, 416012), List(2268, 460193, 93862.6, 5, 469313), 0, List(minCubeSize(107374182400), List(0, 0), List(5, 469313), 0, List(5, 469313), 1, null), null, 0, 1, 5, 0, false, 0, 0, 1771183820917, 1771183824229, 8, 1, null, List(1, 2), null, 8, 8, 502, 0, null, null)"


## VACUUM (Storage Cleanup)

In [0]:
%sql
-- Remove old files no longer needed by Delta Lake
-- Helps reclaim storage space
VACUUM events_table;

path
""


In [0]:
%sql
-- Show the version history of the Delta table again
-- Useful for tracking recent changes
describe history events_table;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
6,2026-02-15T19:31:15.000Z,72977528250978,samyakjain9759545711@gmail.com,VACUUM END,Map(status -> COMPLETED),null,List(2384229307920853),23c0c7c6-9a3b-483e-94c0-d437c97124ad,0215-190144-5stx71h2-v2n,5,SnapshotIsolation,true,"Map(numDeletedFiles -> 0, numVacuumedDirectories -> 1)",null,Databricks-Runtime/18.0.x-aarch64-photon-scala2.13
5,2026-02-15T19:31:14.000Z,72977528250978,samyakjain9759545711@gmail.com,VACUUM START,"Map(retentionCheckEnabled -> true, defaultRetentionMillis -> 604800000)",null,List(2384229307920853),23c0c7c6-9a3b-483e-94c0-d437c97124ad,0215-190144-5stx71h2-v2n,4,SnapshotIsolation,true,"Map(numFilesToDelete -> 0, sizeOfDataToDelete -> 0)",null,Databricks-Runtime/18.0.x-aarch64-photon-scala2.13
4,2026-02-15T19:30:24.000Z,72977528250978,samyakjain9759545711@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> false, clusterBy -> [], zOrderBy -> [""Transaction_ID""], batchId -> 0)",null,List(2384229307920853),81bb862f-eb67-4e1b-b107-8211cf7bd967,0215-190144-5stx71h2-v2n,3,SnapshotIsolation,false,"Map(numRemovedFiles -> 5, numRemovedBytes -> 469313, p25FileSize -> 416012, numDeletionVectorsRemoved -> 1, minFileSize -> 416012, numAddedFiles -> 1, maxFileSize -> 416012, p75FileSize -> 416012, p50FileSize -> 416012, numAddedBytes -> 416012)",null,Databricks-Runtime/18.0.x-aarch64-photon-scala2.13
3,2026-02-15T19:18:18.000Z,72977528250978,samyakjain9759545711@gmail.com,MERGE,"Map(predicate -> [""(Transaction_ID#13608L = Transaction_Id#13585L)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(2384229307920853),14ad4f7b-d803-4395-84b6-7947a6d92215,0215-190144-5stx71h2-v2n,2,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 4, numTargetBytesAdded -> 9120, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 2, executionTimeMs -> 5652, materializeSourceTimeMs -> 341, numTargetRowsInserted -> 2, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 2391, numTargetRowsUpdated -> 2, numOutputRows -> 4, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 4, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 2688)",null,Databricks-Runtime/18.0.x-aarch64-photon-scala2.13
2,2026-02-15T19:03:39.000Z,72977528250978,samyakjain9759545711@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(2384229307920853),1b70e01c-4409-4e71-9c1f-bafb0e5c3add,0215-190144-5stx71h2-v2n,1,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 453352, numDeletionVectorsRemoved -> 0, numOutputRows -> 50000, numOutputBytes -> 460193)",null,Databricks-Runtime/18.0.x-aarch64-photon-scala2.13
1,2026-02-15T14:15:24.000Z,72977528250978,samyakjain9759545711@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(4192937009573805),fc73e082-af42-4a61-865b-0a087dc4e902,0215-134723-t52wdx5u-v2n,0,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 460193, numDeletionVectorsRemoved -> 0, numOutputRows -> 50000, numOutputBytes -> 453352)",null,Databricks-Runtime/18.0.x-aarch64-photon-scala2.13
0,2026-02-15T13:48:45.000Z,7297752